In [1]:
# INSTALLING A REQURIED LIBRARY
!pip install plotly kaleido pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 4.6 MB/s eta 0:00:00


In [2]:
!pip install kaleido==0.2.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 10.8 MB/s eta 0:00:00
  Attempting uninstall: kaleido
    Found existing installation: kaleido 1.3.0
    Uninstalling kaleido-1.3.0:
      Successfully uninstalled kaleido-1.3.0


In [42]:
# INSTALLING DEPENDENCIES
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import warnings
warnings.filterwarnings("ignore")


In [43]:
# FOLDER TO STORE THE CHARTS
os.makedirs("/content/eda_charts", exist_ok=True)

In [44]:
# LOADING DATASET
df = pd.read_csv("/content/clean_matches.csv", parse_dates=["date"])


In [45]:
# ADDING A SCORELINE COLUMN
df["scoreline"] = df["home_score"].astype(str) + "-" + df["away_score"].astype(str)

In [46]:
tier_names = {1: "Major Finals", 2: "Qualifications", 3: "Regional Cups", 4: "Friendlies"}


In [47]:
# INITIAL DATA SUMMARY
print(f"Total matches played: {len(df):,}")
print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Number of unique teams: {df['home_team'].nunique():,}")
print(f"Number of unique tournament: {df['tournament'].nunique():,}")
print(f"Total goals scored: {df['total_goals'].sum():,}")
print(f"Average goals per match: {df['total_goals'].mean():.3f}")


Total matches played: 49,215
Date range: 1872-11-30 -> 2026-03-31
Number of unique teams: 325
Number of unique tournament: 193
Total goals scored: 144,618
Average goals per match: 2.938


### INSPECTING MATCH VOLUME OVER TIME
This gives insight of how the volume of international football has grown over 150% years.

In [48]:
yearly = df.groupby("year").size().reset_index(name = "matches")

In [49]:
yearly_tier = df.groupby(["year", "tournament_tier"]).size().reset_index(name = "matches")

In [50]:
yearly_tier["tier_name"] = yearly_tier["tournament_tier"].map(tier_names)

In [ ]:
# PLOTING MATCH VOLUME OVER TIME


fig1 = make_subplots(
    rows = 2, cols = 1,
    subplot_titles = (
        "Total matches per year (1872-2026)",
        "Matches by tournament tier per year"
    ),
    vertical_spacing = 0.12
)

fig1.add_trace(go.Scatter(
    x = yearly["year"], y = yearly["matches"],
    fill = "tozeroy", line = dict(color = "#185FA5", width = 2),
    fillcolor = "rgba(24, 95, 165, 0.15)", name = "Total matches"
), row = 1, col = 1)


for event, year, y_pos in [("WW1", 1916, 5), ("WW2", 1942, 5), ("COVID", 2020, 400)]:
  fig1.add_vline(x = year, line_dash = "dash", line_color = "#D85A30",
                 line_width = 1, row = 1, col = 1)
  fig1.add_annotation(x = year, y = y_pos, text = event, showarrow = False,
                     font = dict(size = 10, color = "#D85A30"), row = 1,
                     col = 1)

  colors_tier = {
    "Major Finals":  "#D85A30",
    "Qualifications":"#185FA5",
    "Regional Cups": "#1D9E75",
    "Friendlies":    "#B4B2A9"
}


for tier in ["Friendlies", "Regional Cups", "Qualifications", "Major Finals"]:
    subset = yearly_tier[yearly_tier["tier_name"] == tier]
    fig1.add_trace(go.Scatter(
        x = subset["year"], y = subset["matches"],
        name = tier, stackgroup = "one", fill = "tonexty",
        line = dict(color = colors_tier[tier], width = 0.5),
        fillcolor = colors_tier[tier]
    ), row = 2, col = 1)

fig1.update_layout(
    height = 620, showlegend = True,
    title_text = "Match Volume Over Time",
    legend = dict(orientation = "h", y = -0.05),
    paper_bgcolor = "white", plot_bgcolor = "white"
)

fig1.update_xaxes(showgrid = False)
fig1.update_yaxes(gridcolor = "rgba(128,128,128,0.1)")

fig1.write_image("/content/eda_charts/match_volume.png", scale = 1)
fig1.show()


The Match Volume Over Time shows that international football was almost nonexistent before 1900 (or its data was not recorded), and grew slowly through the early 20th century. The graph clearly shows a decrease in match played during two World Wars (1914-18 and 1939-45). The massive increase in number of matches played came post-1950, driven by decolonisation creating dozens of new football nations and the expansion of continental qualification tournaments.

The stacked chart makes clear that Qualifications and Friendlies account for the bluk of this growth. Modern football now sees 1,000 - 2,000+ matches per year compared to near-zero in the 1870s. The COVID dip in 2020 is a sharp one but it recovered immediately as matches were played with empty stadiums.

## RESULT SPLIT INSPECTION (HOME / DRAW / AWAY)
This analysis is used to inspect the home advantage of a team.

In [52]:
result_counts = df["result"].value_counts().reset_index()
result_counts.columns = ["result", "count"]
result_counts["pct"] = (result_counts["count"] / len(df) * 100).round(1)
result_counts["label"] = result_counts["result"].map(
    {"home_win": "Home Win",
     "draw": "Draw",
     "away_win": "Away Win"}
)

In [53]:
neutral_results = df.groupby(["neutral", "result"]).size().reset_index(name = "count")
neutral_results["venue"] = neutral_results["neutral"].map({
    True: "Neutral Venue",
    False: "Home Venue"
})

neutral_results["result_label"] = neutral_results["result"].map(
    {"home_win": "Home Win",
     "draw": "Draw",
     "away_win": "Away Win"}
)

neutral_results["pct"] = neutral_results.groupby("neutral")["count"].transform(
    lambda x: x / x.sum() * 100
).round(1)


In [54]:
tier_results = df.groupby(["tournament_tier", "result"]).size().reset_index(name = "count")
tier_results["tier_name"] = tier_results["tournament_tier"].map(tier_names)

tier_results["result_label"] = tier_results["result"].map(
    {"home_win": "Home Win",
     "draw": "Draw",
     "away_win": "Away Win"}
)

tier_results["pct"] = tier_results.groupby("tournament_tier")["count"].transform(
    lambda x: x / x.sum() * 100
).round(1)


In [ ]:
# PLOT FOR RESULT SPLIT

fig2 = make_subplots(
    rows = 1, cols = 3,
    specs = [[{'type': 'pie'},
              {'type': 'bar'},
              {'type': 'bar'}]],
    subplot_titles = ("All matches", "Home vs Neutral venue",
                      "By tournament tier")
)

fig2.add_trace(go.Pie(
    labels = result_counts['label'], values = result_counts["count"],
    hole = 0.55,
    marker_colors = ["#185FA5", "#888780", "#D85A30"],
    textinfo = "label+percent", showlegend = False
), row = 1, col = 1)

for result, color in [("Home Win", "#185FA5"),
  ("Draw", "#888780"),
  ("Away Win", "#D85A30")]:

  subset = neutral_results[neutral_results["result_label"] == result]
  fig2.add_trace(go.Bar(
      x = subset["venue"], y = subset["pct"],
      name = result, marker_color = color
  ), row = 1, col = 2)

for result, color in [("Home Win", "#185FA5"),
                      ("Draw", "#888780"),
                      ("Away Win", "#D85A30")]:
                      subset = tier_results[tier_results["result_label"] == result]
                      fig2.add_trace(go.Bar(
                          x = subset["tier_name"], y = subset["pct"],
                          name = result, marker_color = color, showlegend = False
                      ), row = 1, col = 3)

fig2.update_layout(
    barmode = "group", height = 440,
    title_text = "Result Split Analysis",
    paper_bgcolor = "white", plot_bgcolor = "white"
)

fig2.update_xaxes(showgrid = False)
fig2.update_yaxes(gridcolor = "rgba(128, 128, 128, 0.1)", ticksuffix = "%")

fig2.write_image("/content/eda_charts/result_split.png", scale=1)
fig2.show()


The vislualization shows that across all international matches, home teams win nearly half the time (~49%), with away wins at just ~28%, which is a massive 21-point gap that clearly isn't random.

In the middle chart: at neutral venues, the home win rate drops to ~45% while away wins jumps to ~34%, almost closing the gap. This is a strong evidence that the advantage is highly venue-driven, not just a reflection of team quality.

Looking at the tournament tier, Qualifications show the biggest home advantage (\~51% home win rate), which is genuine as qualification matches are played in front of the home crowds often against regional opponents. Major Finals show a slightly lower home win rate (~46%), which shows elite teams that participate in major tournaments are less affected by home crowd. The draw rate stays consistent around 22-25% accross all tiers, suggesting draws are more a feature of football itself than of any particular context.

## INSPECTING GOAL DISTRIBUTION

In [68]:
goals_dist = df["total_goals"].value_counts().sort_index().reset_index()
goals_dist.columns = ["goals", "count"]
goals_dist["pct"] = (goals_dist["count"] / len(df) * 100).round(2)

home_goals = df["home_score"].value_counts().sort_index().reset_index()
home_goals.columns = ["goals", "count"]

away_goals = df["away_score"].value_counts().sort_index().reset_index()
away_goals.columns = ["goals", "count"]

In [ ]:

fig3 = make_subplots(
    rows = 1, cols = 2,
    subplot_titles = (
        "Total goals per match distribution",
        "Home vs away goals scored distribution"
    )
)

fig3.add_trace(go.Bar(
    x = goals_dist[goals_dist["goals"] <= 12]["goals"],
    y = goals_dist[goals_dist["goals"] <= 12]["count"],
    marker_color = ["#185FA5" if g == 2 else "#B5D4F4"
                  for g in goals_dist[goals_dist["goals"] <= 12]["goals"]],
    name = "Total goals", showlegend = False,
    text = goals_dist[goals_dist["goals"] <= 12]["pct"].apply(lambda x: f"{x}%"),
    textposition = "outside", textfont_size = 9
), row = 1, col = 1)


fig3.add_trace(go.Bar(
    x = home_goals[home_goals["goals"] <= 10]["goals"],
    y = home_goals[home_goals["goals"] <= 10]["count"],
    name = "Home goals", marker_color = "#185FA5", opacity = 0.7
), row = 1, col = 2)

fig3.add_trace(go.Bar(
    x = away_goals[away_goals["goals"] <= 10]["goals"],
    y = away_goals[away_goals["goals"] <= 10]["count"],
    name = "Away goals", marker_color = "#D85A30", opacity = 0.7
), row = 1, col = 2)


fig3.update_layout(
    barmode = "overlay", height = 440,
    title_text = "Goals Distribution",
    paper_bgcolor = "white", plot_bgcolor = "white"
)

fig3.update_xaxes(showgrid = False, title_text = "Goals")
fig3.update_yaxes(gridcolor = "rgba(128,128,128,0.1)", title_text = "Matches")

fig3.write_image("/content/eda_charts/goals_distribution.png", scale = 1)
fig3.show()

Goals distribution is strongly right-skewed with 2-goal matches being the mode at 22%, and roughly 73% of all games producing 3 or fewer total goals which confirms that football is a fundamentally low-scoring sport. The 0-goal frequency at 8% is worth noting. Goalless games are more common than any result involving 5+ goals.

On the home vs away split, away teams score 0 goals far more frequently than home teams, while home teams dominate the 1–2 goal bins which is a clear statistical fingerprint of away sides prioritising defensive solidity over attack.

Both distributions converge at higher goal counts which means venue effect essentially disappears in high-scoring games. For modelling purposes, this skew justifies log-compression of goal margins rather than treating each additional goal as linearly equal in value.

## INSPECTING MOST COMMON SCORELINES

In [70]:
top_scorelines = df["scoreline"].value_counts().head(15).reset_index()
top_scorelines.columns = ["scoreline", "count"]

top_scorelines["home_s"] = top_scorelines["scoreline"].apply(lambda x: int(x.split("-")[0]))
top_scorelines["away_s"] = top_scorelines["scoreline"].apply(lambda x: int(x.split("-")[1]))

top_scorelines["result_type"] = top_scorelines.apply(
    lambda r: "Home Win" if r["home_s"] > r["away_s"]
    else ("Away Win" if r["away_s"] > r["home_s"] else "Draw"), axis = 1
)
color_map = {"Home Win": "#185FA5", "Away Win": "#D85A30", "Draw": "#888780"}
top_scorelines["color"] = top_scorelines["result_type"].map(color_map)


In [ ]:
fig4 = go.Figure(go.Bar(
    x = top_scorelines["scoreline"],
    y = top_scorelines["count"],
    marker_color = top_scorelines["color"],
    text = top_scorelines["count"].apply(lambda x: f"{x:,}"),
    textposition = "outside",
    customdata = top_scorelines["result_type"],
    hovertemplate = "Scoreline: %{x}<br>Matches: %{y:,}<br>Type: %{customdata}<extra></extra>"
))

for result, color, x in [("Home Win","#185FA5",0.72), ("Draw","#888780",0.82), ("Away Win","#D85A30",0.92)]:
    fig4.add_annotation(x = x, y = 1.06, xref = "paper", yref = "paper",
                        text = f"{result}", showarrow = False,
                        font = dict(color = color, size = 12))

fig4.update_layout(
    title = "Top 15 Most Common Scorelines",
    xaxis_title = "Scoreline (home-away)", yaxis_title = "Number of matches",
    height = 470, showlegend = False,
    xaxis = dict(showgrid = False),
    yaxis = dict(gridcolor = "rgba(128,128,128,0.1)"),
    paper_bgcolor = "white", plot_bgcolor = "white"
)

fig4.write_image("/content/eda_charts/scorelines.png", scale = 1)
fig4.show()

The graph shows, 1-0 is the most common scoreline in international football history at 5,079 matches, which alone tells everything about how decisive single goals are in this sport. The top 6 scorelines all involve 2 or fewer total goals, reinforcing the low-scoring nature confirmed in the distribution analysis. What stands out is the asymmetry between equivalent home and away scorelines. 2-0 (3,823) appears significantly more often than its away equivalent 0-2 (2,216), and 2-1 (3,758) vs 1-2 (2,541) shows the same pattern, quantifying home advantage directly at the scoreline level.

The 0-0 result at 3,956 is notably the third most common outcome, appearing more frequently than any scoreline involving 3+ total goals, which again highlights how often defensive structures hold completely. For a rating model, the dominance of 1-goal margins in the top scorelines justifies high sensitivity to single-goal results rather than weighting larger margins heavily.

# INSPECTING HOME ADVANTAGE TREND BY DECADE

In [73]:
home_only = df[df["neutral"] == False].copy()
neutral_only = df[df["neutral"] == True].copy()

ha_decade = home_only.groupby("decade").agg(
    matches = ("result", "count"),
    home_win_pct = ("result", lambda x: round((x == "home_win").mean() * 100, 1)),
    away_win_pct = ("result", lambda x: round((x == "away_win").mean() * 100, 1)),
    draw_pct = ("result", lambda x: round((x == "draw").mean() * 100, 1)),
).reset_index()

ha_decade["gap"] = (ha_decade["home_win_pct"] - ha_decade["away_win_pct"]).round(1)

ha_neutral = neutral_only.groupby("decade").agg(
    matches = ("result", "count"),
    home_win_pct = ("result", lambda x: round((x == "home_win").mean() * 100, 1)),
    away_win_pct = ("result", lambda x: round((x == "away_win").mean() * 100, 1)),
).reset_index()


In [ ]:
fig5 = make_subplots(
    rows = 2, cols = 2,
    subplot_titles = (
        "Home win % vs Away win % by decade",
        "Home advantage gap (home% − away%) over time",
        "Draw % trend by decade",
        "Neutral venue"
    ),
    vertical_spacing = 0.15, horizontal_spacing = 0.1
)

fig5.add_trace(go.Scatter(x = ha_decade["decade"], y = ha_decade["home_win_pct"],
    name = "Home win %", line = dict(color = "#185FA5", width = 2.5),
    mode = "lines+markers", marker = dict(size = 6)), row = 1, col = 1)

fig5.add_trace(go.Scatter(x = ha_decade["decade"], y = ha_decade["away_win_pct"],
    name = "Away win %", line = dict(color = "#D85A30", width = 2.5, dash = "dash"),
    mode = "lines+markers", marker = dict(size = 6)), row = 1, col = 1)

fig5.add_trace(go.Bar(x = ha_decade["decade"], y = ha_decade["gap"],
    marker_color = ["#185FA5" if g > 20 else "#B5D4F4" for g in ha_decade["gap"]],
    name = "Advantage gap", showlegend = False), row = 1, col = 2)

fig5.add_trace(go.Scatter(x = ha_decade["decade"], y = ha_decade["draw_pct"],
    name = "Draw %", line = dict(color = "#888780", width = 2),
    fill = "tozeroy", fillcolor = "rgba(136,135,128,0.1)",
    mode = "lines+markers", marker = dict(size = 6)), row = 2, col = 1)

if len(ha_neutral) > 0:
    fig5.add_trace(go.Scatter(x = ha_neutral["decade"], y = ha_neutral["home_win_pct"],
        name = "'Home' at neutral", line = dict(color = "#1D9E75", width = 2),
        mode = "lines+markers"), row = 2, col = 2)

    fig5.add_trace(go.Scatter(x = ha_neutral["decade"], y = ha_neutral["away_win_pct"],
        name = "'Away' at neutral", line = dict(color = "#BA7517", width = 2, dash = "dash"),
        mode = "lines+markers"), row = 2, col = 2)

fig5.update_layout(
    height = 620, title_text = "Home Advantage Analysis",
    legend = dict(orientation = "h", y = -0.08),
    paper_bgcolor = "white", plot_bgcolor = "white"
)

fig5.update_xaxes(showgrid = False)
fig5.update_yaxes(gridcolor = "rgba(128,128,128,0.1)")

fig5.write_image("/content/eda_charts/home_advantage.png", scale = 1)
fig5.show()

Home win rate has sat rock-solid around 50% for over a century while away wins settled at 25–27%, producing a persistent ~24-point gap that hasn't meaningfully budged since 1950, which shows home advantage in international football isn't shrinking, it's structural. The rising draw rate from 9% to 23% over the same period tells the tactical story: those draws are largely coming at the expense of away wins as defensive organisation improved.

The neutral venue chart is the most convincing piece of evidence. Strip away the home ground and that gap almost halves, with both sides converging toward 35–40% win rates. That's not a coincidence, that's proof the advantage is venue-driven, not team-quality-driven, and any rating model that doesn't account for it with an explicit home bonus will be systematically wrong.

## INSPECTING TOP TEAM BY WIN RATE

In [56]:
home_m = df.groupby("home_team").agg(hm=("result","count"),
                                     hw=("result", lambda x:(x=="home_win").sum())).reset_index().rename(columns={"home_team":"team"})

away_m = df.groupby("away_team").agg(am=("result","count"),
                                     aw=("result", lambda x:(x=="away_win").sum())).reset_index().rename(columns={"away_team":"team"})

# hm = home match played, hw = home match won

In [57]:
teams_df = home_m.merge(away_m, on = "team", how = "outer").fillna(0)
teams_df["total"] = teams_df["hm"] + teams_df["am"]
teams_df["wins"] = teams_df["hw"] + teams_df["aw"]
teams_df["win_rate"] = (teams_df["wins"] / teams_df["total"] * 100).round(1)

In [58]:
teams_df.head()

,team,hm,hw,am,aw,total,wins,win_rate
0,Abkhazia,22.0,11.0,10.0,3.0,32.0,14.0,43.8
1,Afghanistan,48.0,18.0,96.0,17.0,144.0,35.0,24.3
2,Albania,208.0,78.0,188.0,32.0,396.0,110.0,27.8
3,Alderney,48.0,1.0,87.0,4.0,135.0,5.0,3.7
4,Algeria,354.0,206.0,261.0,80.0,615.0,286.0,46.5


In [59]:
# GOALS PER TEAM
home_gf = df.groupby("home_team")["home_score"].sum().rename("gf")
home_ga = df.groupby("home_team")["away_score"].sum().rename("ga")
away_gf = df.groupby("away_team")["away_score"].sum().rename("gf")
away_ga = df.groupby("away_team")["home_score"].sum().rename("ga")
team_gf = (home_gf.add(away_gf, fill_value = 0)).reset_index()
team_gf.columns = ["team", "goals_for"]
team_ga = (home_ga.add(away_ga, fill_value = 0)).reset_index()
team_ga.columns = ["team", "goals_against"]
teams_df = teams_df.merge(team_gf, on = "team").merge(team_ga, on = "team")
teams_df["goal_diff"] = teams_df["goals_for"] - teams_df["goals_against"]
teams_df["gpg"] = (teams_df["goals_for"] / teams_df["total"]).round(2)


In [60]:
top_wr_100  = teams_df[teams_df["total"] >= 100].nlargest(20, "win_rate")
top_wr_500  = teams_df[teams_df["total"] >= 500].nlargest(15, "win_rate")
top_goals   = teams_df[teams_df["total"] >= 100].nlargest(15, "goals_for")


In [ ]:
# PLOTTING
fig6 = make_subplots(rows = 1, cols = 2,
                     subplot_titles = ("Top 20 teams by win rate (min 100 matches)",
                                     "Top 15 teams by total goals scored (min 100 matches)"))

major = {"Brazil","Spain","Germany","England","Argentina","France","Italy",
         "Netherlands","Portugal","Uruguay"}
colors_wr = ["#185FA5" if t in major else "#B5D4F4" for t in top_wr_100["team"]]

fig6.add_trace(go.Bar(
    y = top_wr_100["team"], x = top_wr_100["win_rate"],
    orientation = "h", marker_color = colors_wr,
    text = top_wr_100["win_rate"].apply(lambda x: f"{x}%"),
    textposition = "outside",
    customdata = top_wr_100["total"].astype(int),
    hovertemplate = "%{y}: %{x}% win rate<br>%{customdata} matches<extra></extra>"
), row = 1, col = 1)

fig6.add_trace(go.Bar(
    y = top_goals["team"], x = top_goals["goals_for"],
    orientation = "h", marker_color = "#1D9E75",
    text = top_goals["goals_for"].astype(int),
    textposition = "outside"
), row = 1, col = 2)

fig6.update_layout(height = 520, title_text="Team Performance Overview",
                   showlegend = False)
fig6.update_xaxes(showgrid = False)
fig6.update_yaxes(autorange = "reversed", gridcolor= "rgba(128,128,128,0.1)")
fig6.write_html("eda_charts/top_teams.png")
fig6.show()


Brazil sits at the top of the win rate chart among serious footballing nations at \~63.5%, followed closely by Spain (\~58.9%) and Germany (\~58%), which lines up exactly with what you'd expect from football history. Jersey and Guernsey appearing above Brazil is a statistical quirk as they've played just over 100 matches, mostly against weak opposition, so their win rate is inflated and not really comparable.

Iran at 56.9% is probably the most surprising legitimate entry, showing they've been quietly dominant in Asian football for decades.

The goals chart tells a slightly different story. England leads with 2,370+ goals simply because they've been playing international football the longest and have played an enormous number of matches. Sweden appearing fourth ahead of Argentina is again partly a volume effect from their heavy friendly schedule. What stands out is that Brazil, Germany, and Argentina all combine high win rates and high goal counts, which is the real mark of an elite footballing nation as they don't just win, they score prolifically doing it. Spain being lower on the goals chart despite a high win rate suggests they win more through control and efficiency than through high-scoring games, which fits their historical style perfectly.

## INSPECTING MOST ACTIVE NATION
The purpose of this inspection is that the teams that have played more matches have better-converged, and thus have more trustworthy ratings.

In [62]:
home_count = df.groupby("home_team").size().rename("home_matches")
away_count = df.groupby("away_team").size().rename("away_matches")

activity = pd.concat([home_count, away_count], axis = 1).fillna(0).astype(int)
activity["total_matches"] = activity["home_matches"] + activity["away_matches"]
activity = activity.reset_index().rename(columns = {"index": "team"})
activity = activity.sort_values("total_matches", ascending = False).head(25)

activity = activity.merge(teams_df[["team", "win_rate", "goals_for"]],
                          on = "team", how = "left")

In [ ]:
# PLOTING THE RESULTS

fig7 = make_subplots(rows = 1, cols = 2,
                     subplot_titles = (
                         "Top 25 most active nations",
                         "Activity vs win rate"
                     ))

fig7.add_trace(go.Bar(
    y = activity["team"], x = activity["total_matches"],
    orientation = "h",
    marker_color = ["#185FA5" if t in major
                    else "#B5D4F4" for t in activity["team"]],
    text = activity["total_matches"], textposition = "outside"
), row = 1, col = 1)

fig7.add_trace(go.Scatter(
    x = activity["total_matches"], y = activity["win_rate"],
    mode = "markers+text",
    text = activity["team"],
    textposition = "top center",
    textfont = dict(size = 9),
    marker = dict(size = 10,
                  color = ["#185FA5" if t in major
                           else "#888780" for t in activity["team"]]
                  ),
    hovertemplate="%{text}<br>Matches: %{x}<br>Win rate: %{y}%<extra></extra>"
), row = 1, col = 2)

fig7.update_layout(
    height = 540, title_text = "Most Active Nations",
    showlegend = False,
    paper_bgcolor = "white", plot_bgcolor = "white"
)

fig7.update_xaxes(showgrid=False)
fig7.update_yaxes(autorange="reversed", gridcolor="rgba(128,128,128,0.1)")

fig7.write_image("/content/eda_charts/active_nations.png", scale=2)
fig7.show()


The graph shows that, Sweden tops the activity char with 1099 matches, which is surprising for a nation not typically mentioned in the same level as Brazil or Germany. This is largely because of their historically heavy friendly schedule. England, Argentina, Brazil, and Germany all cluster tightly between 1020 - 1088 matches, which is reasonable given their long footballing histories.

The scatter plot on the right is more interesting as it answers the question "Does playing more matches make you better?", and the answer is clearly no. Finland, Zambia, and Japan have played 800+ matches but sits at 24-50% win rates, while Brazil played similar volumne but wins nearly 63% of the time. The top-right cluster of Brazip, England, Argentina and Germany is the sweet spot with high activity and high win rate, confirming that these are elite nations.

## ANALYSING GOALS PER DECADE TREND

In [ ]:
decade_goals = df.groupby("decade").agg(
    matches = ("total_goals", "count"),
    avg_total = ("total_goals", "mean"),
    avg_home = ("home_score", "mean"),
    avg_away = ("away_score", "mean"),
    pct_0_0 = ("total_goals", lambda x: round((x == 0). mean() * 100, 1)),
    pct_high = ("total_goals", lambda x: round((x >= 4).mean() * 100, 1)),
).reset_index().round(2)

fig8 = make_subplots(
    rows = 2, cols = 2,
    subplot_titles=(
        "Average goals per match by decade",
        "Home vs away avg goals per decade",
        "% of matches with 4+ goals (high-scoring)",
        "% of matches ending 0-0"
    ),
    vertical_spacing = 0.15
)

fig8.add_trace(go.Bar(
    x = decade_goals["decade"], y = decade_goals["avg_total"],
    marker_color = ["#185FA5" if v == decade_goals["avg_total"].min() else
                  "#D85A30" if v == decade_goals["avg_total"].max() else "#B5D4F4"
                  for v in decade_goals["avg_total"]],
    text = decade_goals["avg_total"].round(2), textposition = "outside",
    name = "Avg goals"
), row = 1, col = 1)

fig8.add_trace(go.Scatter(x = decade_goals["decade"], y = decade_goals["avg_home"],
    name = "Home goals", line = dict(color = "#185FA5", width = 2.5),
    mode = "lines+markers"), row = 1, col = 2)

fig8.add_trace(go.Scatter(x = decade_goals["decade"], y = decade_goals["avg_away"],
    name = "Away goals", line = dict(color = "#D85A30", width = 2.5, dash = "dash"),
    mode = "lines+markers"), row = 1, col = 2)

fig8.add_trace(go.Bar(
    x = decade_goals["decade"], y = decade_goals["pct_high"],
    marker_color = "#1D9E75", name = "High scoring %",
    text = decade_goals["pct_high"].apply(lambda x: f"{x}%"),
    textposition = "outside"
), row = 2, col = 1)

fig8.add_trace(go.Scatter(
    x = decade_goals["decade"], y = decade_goals["pct_0_0"],
    fill = "tozeroy", fillcolor = "rgba(136,135,128,0.15)",
    line = dict(color = "#888780", width = 2),
    mode = "lines+markers", name = "0-0 %"
), row = 2, col = 2)

fig8.update_layout(
    height = 620, title_text = "Goals Trend Analysis",
    legend = dict(orientation = "h", y = -0.08),
    paper_bgcolor = "white", plot_bgcolor = "white"
)

fig8.update_xaxes(showgrid = False)
fig8.update_yaxes(gridcolor = "rgba(128,128,128,0.1)")

fig8.write_image("/content/eda_charts/goals_trend.png", scale=2)
fig8.show()





The data tells a clear story of football's tactical evolution. Average goals per game have fallen from a chaotic 5.58 in the 1880s to a stable ~2.7 post-1990, which high-scoring matches (4+ goals) dropping from 62% to under 30% over the same period.

Home teams have consistently outscored away teams in every decade, but away goals have fallen harder, suggesting away sides grew increasingly defensive over time.

The rise in 0-0 draws from under 2% to nearly 10% in the modern era confirms this shift. Essentially, football went from an unstructured attacking free-for-all to a tactically disciplined, low-scoring game, and the 1950s-70s is where that transition clearly happened.

## TOURNAMENT BREAKDOWN
Not all matches are created equal. A World Cup final should shift ratings far more than a friendly played during an international break.

In [65]:
tier_stats = df.groupby("tournament_tier").agg(
    matches = ("result","count"),
    home_wr = ("result", lambda x: round((x == "home_win").mean() * 100, 1)),
    away_wr = ("result", lambda x: round((x == "away_win").mean() * 100, 1)),
    draw_r = ("result", lambda x: round((x == "draw").mean() * 100, 1)),
    avg_goals = ("total_goals","mean"),
    shootouts = ("went_to_shootout","sum"),
).reset_index()

tier_stats["tier_name"] = tier_stats["tournament_tier"].map(tier_names)
tier_stats["avg_goals"] = tier_stats["avg_goals"].round(2)


In [66]:
top_tournaments = df["tournament"].value_counts().head(12).reset_index()
top_tournaments.columns = ["tournament","matches"]
top_tournaments["tier"] = top_tournaments["tournament"].map(
    df.drop_duplicates("tournament").set_index("tournament")["tournament_tier"]
)
top_tournaments["tier_name"] = top_tournaments["tier"].map(tier_names)


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig9 = make_subplots(
    rows = 3, cols = 2,
    subplot_titles = (
        "Matches per tournament tier",
        "Home win % by tier",
        "Average goals by tier",
        "",
        "Top 12 individual tournaments by match count",
        ""
    ),
    specs = [
        [{}, {}],
        [{}, None],
        [{"colspan": 2}, None]
    ],
    vertical_spacing = 0.12
)


fig9.add_trace(go.Bar(
    x = tier_stats["tier_name"],
    y = tier_stats["matches"],
    marker_color = ["#D85A30","#185FA5","#1D9E75","#B4B2A9"],
    text = tier_stats["matches"].apply(lambda x: f"{x:,}"),
    textposition = "outside",
    name = "Matches"
), row = 1, col = 1)


for result, col, color in [
    ("home_wr", "Home Win", "#185FA5"),
    ("away_wr", "Away Win", "#D85A30"),
    ("draw_r", "Draw", "#888780")
]:
    fig9.add_trace(go.Bar(
        x = tier_stats["tier_name"],
        y = tier_stats[result],
        name = col,
        marker_color = color
    ), row = 1, col = 2)


fig9.add_trace(go.Bar(
    x = tier_stats["tier_name"],
    y = tier_stats["avg_goals"],
    marker_color = ["#D85A30","#185FA5","#1D9E75","#B4B2A9"],
    text = tier_stats["avg_goals"],
    textposition = "outside",
    name = "Avg goals",
    showlegend = False
), row = 2, col = 1)


t_colors = {
    "Major Finals": "#D85A30",
    "Qualifications": "#185FA5",
    "Regional Cups": "#1D9E75",
    "Friendlies": "#B4B2A9"
}


fig9.add_trace(go.Bar(
    y = top_tournaments["tournament"],
    x = top_tournaments["matches"],
    orientation = "h",
    marker_color = [
        t_colors.get(t, "#888780")
        for t in top_tournaments["tier_name"]
    ],
    text = top_tournaments["matches"].apply(lambda x: f"{x:,}"),
    textposition = "outside",
    showlegend = False
), row = 3, col = 1)

fig9.update_layout(
    barmode = "group",
    height = 850,
    title_text = "Tournament Breakdown",
    legend = dict(
        orientation = "h",
        y = -0.05
    ),
    paper_bgcolor = "white",
    plot_bgcolor = "white"
)

fig9.update_xaxes(showgrid = False)
fig9.update_yaxes(
    gridcolor = "rgba(128,128,128,0.1)"
)

fig9.write_image("/content/eda_charts/tournaments.png", scale = 1)
fig9.show()

Friendlies dominate raw match volume at \~19000 games, but that number is somewhat misleading as they matter least competitively. FIFA World Cup qualification is by far the biggest competitive competition with 8,771 matches, surpasing the UEFA Euro qualification (\~2,824) and AFCON qualification (\~2327).

The qualification matches show the strongest home advantage, which is reasonable given they're played in home stadiums with high-pressure local crowds. Major Finals, often held at neutral venues, shows the lowest home advantage at around 46%.

Averages goals, however, is similar across all tiers, hovering between 2.73 and 2.88, suggesting tournament prestige doesn't dramatically change how many goals are scored, it just changes who wins.

Hence, for any rating model,  all matches should not be weighted equally, and the volume of friendlies in the dataset means they need a lower K-factor, otherwise they will dominate out the signal from meaningful competitive games.